**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB06 — Activation Patching Sweep and Candidate Selection

**Track B · Mechanistic interpretability of SEL-4L · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | Which internal activation, at which layer and position, carries (a) the difference that the identity input makes and (b) the difference that the context makes to Δ? |
| **Experiment** | E12 |
| **Population** | identity population (P-player), validation. Candidates are **selected on the discovery half** and **measured on the estimation half**. |
| **Outputs** | `results/MB06_patching.parquet`, `results/MB06_site_summary.csv`, `config/candidates.json` (fingerprinted before estimation), `figures/MB06_*` |
| **Module written** | `src/patching.py` |
| **Test data** | none |



In [1]:
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = os.environ.get("MI_CPU_THREADS", "4")
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/aayush/code/soccer_mi
Track A (read only): /home/aayush/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/aayush/code/soccer_mi/.mi_analysis_rerun


In [2]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/patching.py').exists(), "Missing dependency: src/patching.py"

In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/aayush/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [4]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())

partners = pd.read_parquet(POP / "swap_partners.parquet")
donors = pd.read_parquet(POP / "context_donors.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
population = val_conflicts[val_conflicts[IDENTITY] > 0].reset_index(drop=True)
population = population.merge(donors[["event_row", "donor_window"]], on="event_row", how="left")
SITES = patching.sweep_sites(N_LAYERS, N_HEADS)
print(f"population: {len(population):,} | sites: {len(SITES)} -> {[label for label, _, _ in SITES]}")

/home/aayush/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 49 | primary seed 103 | identity variable: player
half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651
population: 1,744 | sites: 21 -> ['embedding', 'L0 attn_out', 'L0 head 0', 'L0 head 1', 'L0 mlp_out', 'L0 resid_post', 'L1 attn_out', 'L1 head 0', 'L1 head 1', 'L1 mlp_out', 'L1 resid_post', 'L2 attn_out', 'L2 head 0', 'L2 head 1', 'L2 mlp_out', 'L2 resid_post', 'L3 attn_out', 'L3 head 0', 'L3 head 1', 'L3 mlp_out', 'L3 resid_post']


## 1. Terms

| Term | Meaning here |
|---|---|
| **Activation patching** | Run the model on two inputs that differ in one respect, then rerun one of them with a single internal activation copied from the other. If the output moves, that activation carries the difference. |
| **Clean / corrupted run** | The event as recorded / the event with one thing changed. |
| **Denoising** | Clean activation into the corrupted run: is this activation *sufficient* to bring the clean behaviour back? |
| **Noising** | Corrupted activation into the clean run: is this activation *necessary* for the clean behaviour? |
| **Pooled recovery R** | Σ sign(d_i) c_i / Σ |d_i| with d_i = (m_clean − m_corrupted)_i and c_i = (m_patched − m_corrupted)_i for denoising or (m_clean − m_patched)_i for noising: the share of the clean-versus-corrupted difference in Δ that one patched activation restores (denoising) or removes (noising). Pooling over events avoids dividing by near-zero differences of single events (Zhang & Nanda 2024); aligning each event by the sign of its own difference keeps opposite-signed events from cancelling in the denominator. R = 1: fully restored; R = 0: nothing restored. |
| **Aligned change** | sign(d_i) × c_i: positive when a denoising patch moves Δ back towards the event's own clean value (or a noising patch moves it away), whichever direction that is. |

## 2. Contrasts, sites and positions (fixed in the pre-registration)

| Contrast | Clean | Corrupted | What differs |
|---|---|---|---|
| identity removal (primary) | the event as recorded | identity index → 0 at the target | the identity vector at one position |
| identity swap (secondary, player only) | as recorded | player → matched partner | which player |
| context | context X with player A (as recorded) | the donor window Y (same role and scoring position, other match) with player A written at the target | everything except the target's player |

The metric in all runs is Δ computed with the clean event's (a_ctx, a_id).

**Sites (21):** the embedding; and for each of the 4 layers, the attention output, each of the 2 heads, the MLP output and the residual stream leaving the block.

**Positions.** For the identity contrasts only the **target** position is patched: the inputs differ only there, every earlier position is identical in both runs (causal mask), and later positions cannot influence the target's prediction. For the context contrast every position from 15 events before the target up to the target is patched.

## 3. Built-in checks before the sweep

1. **No-op patch:** copying the clean run's own activation into the clean run must change Δ by exactly 0.
2. **Index-0 rows under identity removal:** clean and corrupted inputs are identical, so every site must give exactly 0.

In [5]:
check_table = val_conflicts[val_conflicts[IDENTITY] == 0].head(BATCH)
with torch.no_grad():
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, check_table, BATCH):
        corrupted = interventions.remove_feature(inputs, IDENTITY, positions)
        m_clean, m_corrupt, patched = patching.patch_batch(MODEL, inputs, corrupted, positions, a_ctx, a_id, SITES, [0], ["denoise"])
        largest = max(float(np.abs(values - m_corrupt).max()) for values in patched.values())
        print(f"index-0 rows: largest change from any patch {largest} and clean-corrupted difference {float(np.abs(m_clean - m_corrupt).max())}")
        assert largest == 0.0
        m_clean, _, patched = patching.patch_batch(MODEL, inputs, inputs, positions, a_ctx, a_id, SITES, [0], ["denoise"])
        largest = max(float(np.abs(values - m_clean).max()) for values in patched.values())
        print(f"no-op patch: largest change {largest}")
        assert largest == 0.0
        break

index-0 rows: largest change from any patch 0.0 and clean-corrupted difference 0.0


no-op patch: largest change 0.0


## 4. The sweep

Every contrast × direction × site × position is run for every event of the population (both halves). One row per event per combination is stored.

In [6]:
def corrupted_inputs(contrast, inputs, positions, chunk):
    if contrast == "identity removal":
        return interventions.remove_feature(inputs, IDENTITY, positions)
    if contrast == "identity swap":
        partner = torch.as_tensor([partner_of[p] if partner_of.get(p, -1) > 0 else p for p in chunk["player"]], device=DEVICE)
        return interventions.set_feature(inputs, "player", positions, partner)
    donor_inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["donor_window"].to_numpy())
    return interventions.write_identity(donor_inputs, inputs, positions, ["player"])

CONTRASTS = {"identity removal": [0], "context": list(range(-15, 1))}
if IDENTITY == "player":
    CONTRASTS["identity swap"] = [0]
records = []
start = time.time()
with torch.no_grad():
    for contrast, offsets in CONTRASTS.items():
        table = population if contrast != "context" else population[population["donor_window"].notna()].reset_index(drop=True)
        if contrast == "context":
            table["donor_window"] = table["donor_window"].astype(int)
        for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, table, BATCH):
            chunk = table.iloc[part]
            corrupted = corrupted_inputs(contrast, inputs, positions, chunk)
            m_clean, m_corrupt, patched = patching.patch_batch(MODEL, inputs, corrupted, positions, a_ctx, a_id, SITES,
                                                               offsets, ["denoise", "noise"])
            for (direction, label, offset), values in patched.items():
                records.append(pd.DataFrame({"event_row": chunk["event_row"].to_numpy(), "match_id": chunk["match_id"].to_numpy(),
                                             "half": chunk["half"].to_numpy(), "contrast": contrast, "direction": direction,
                                             "site": label, "offset": offset, "m_clean": m_clean, "m_corrupt": m_corrupt,
                                             "m_patched": values}))
        print(f"{contrast}: {len(table):,} events, {len(offsets)} positions, {time.time() - start:.0f} s elapsed")
sweep = pd.concat(records, ignore_index=True)
sweep["raw_change"] = np.where(sweep["direction"] == "denoise", sweep["m_patched"] - sweep["m_corrupt"],
                               sweep["m_clean"] - sweep["m_patched"])
sweep["aligned_change"] = mi_metrics.aligned(sweep["raw_change"], sweep["m_clean"] - sweep["m_corrupt"])
sweep.to_parquet(OUT / "results" / "MB06_patching.parquet", index=False)
print(f"stored {len(sweep):,} rows")

identity removal: 1,744 events, 1 positions, 32 s elapsed


context: 1,743 events, 16 positions, 508 s elapsed


identity swap: 1,744 events, 1 positions, 540 s elapsed


stored 1,317,792 rows


## 5. Candidate selection on the discovery half (pre-registered rule)

For each contrast × direction × site × position: the match-bootstrap interval of the **aligned** change sign(m_clean − m_corrupted) × (m_patched − m_corrupted), and a two-sided bootstrap p-value. The aligned change is used because removing or swapping the player moves Δ up on some events and down on others; the signed raw change would average such movements towards 0 even at a site that fully carries them. Benjamini–Hochberg at q = 0.05 within each contrast × direction. A site-position is a **candidate** if it passes in **denoising** for the primary identity contrast (identity removal) or for the context contrast. The list is written to `config/candidates.json` and fingerprinted **before** the estimation half is evaluated; lists at q = 0.01 and 0.10, and the q = 0.05 list restricted to pooled recovery ≥ 0.10 (a provisional size filter), are saved for sensitivity only. Significance says a site's effect is reliably non-zero, not that it is large; the pooled recovery column shows the size.

In [7]:
discovery = sweep[sweep.half == "discovery"]
selection_rows = []
for (contrast, direction, site, offset), group in discovery.groupby(["contrast", "direction", "site", "offset"], sort=False):
    summary = mi_metrics.summarise(group["aligned_change"], group["match_id"], N_BOOT)
    pooled = mi_metrics.pooled_recovery(group["raw_change"], group["m_clean"] - group["m_corrupt"], group["match_id"], N_BOOT)
    selection_rows.append({"contrast": contrast, "direction": direction, "site": site, "offset": offset, "n": summary["n"],
                           "mean raw change": group["raw_change"].mean(), "mean aligned change": summary["mean"],
                           "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                           "p": mi_metrics.bootstrap_pvalue(group["aligned_change"], group["match_id"], N_BOOT),
                           "pooled recovery": pooled["recovery"]})
selection = pd.DataFrame(selection_rows)
lists = {}
for q in [0.01, 0.05, 0.10]:
    chosen = []
    for (contrast, direction), group in selection.groupby(["contrast", "direction"]):
        reject, adjusted = mi_metrics.benjamini_hochberg(group["p"].to_numpy(), q)
        selection.loc[group.index, f"q-value"] = adjusted
        if direction == "denoise" and contrast in ["identity removal", "context"]:
            for index in group.index[reject]:
                row = selection.loc[index]
                chosen.append({"contrast": row["contrast"], "site": row["site"], "offset": int(row["offset"]),
                               "q": float(row["q-value"]), "mean_aligned_change": float(row["mean aligned change"]),
                               "pooled_recovery": float(row["pooled recovery"])})
    lists[str(q)] = chosen
large = [item for item in lists["0.05"] if item["pooled_recovery"] >= 0.10]
candidates = {"rule": prereg["candidate_rule_MB06"], "q": 0.05, "candidates": lists["0.05"],
              "sensitivity": {"0.01": lists["0.01"], "0.1": lists["0.1"],
                              "q 0.05 and pooled recovery >= 0.10 (provisional)": large},
              "discovery_matches": prereg["discovery_matches"], "written_at": fingerprint.now()}
mi_setup.save_json(OUT / "config" / "candidates.json", candidates)
fingerprint.record_files(MANIFEST, {"config/candidates.json": OUT / "config" / "candidates.json"})
print(f"candidates at q = 0.05: {len(lists['0.05'])} | q = 0.01: {len(lists['0.01'])} | q = 0.10: {len(lists['0.1'])} | "
      f"q = 0.05 with pooled recovery >= 0.10 (provisional, sensitivity only): {len(large)}")
print(pd.DataFrame(lists["0.05"]).to_string(index=False) if lists["0.05"] else "no candidate selected")

candidates at q = 0.05: 36 | q = 0.01: 14 | q = 0.10: 42 | q = 0.05 with pooled recovery >= 0.10 (provisional, sensitivity only): 18
        contrast          site  offset        q  mean_aligned_change  pooled_recovery
         context     embedding      -9 0.020990             0.007729         0.001117
         context     embedding       0 0.020990             6.718747         0.971101
         context     L0 head 0     -13 0.033583             0.001210         0.000175
         context    L0 mlp_out      -8 0.020990             0.007259         0.001049
         context    L0 mlp_out      -7 0.020990             0.009390         0.001357
         context    L0 mlp_out       0 0.020990             4.457503         0.644269
         context L0 resid_post       0 0.020990             6.864261         0.992133
         context   L1 attn_out     -15 0.045795             0.000390         0.000056
         context   L1 attn_out      -3 0.033583            -0.004031        -0.000583
       

## 6. Estimation of the candidates' effects (estimation half)

Pooled recovery and raw change with match-bootstrap intervals, in both directions, for every candidate. Agreement between denoising and noising is a sign that the site carries the information without a substitute; disagreement suggests redundant components.

In [8]:
estimation = sweep[sweep.half == "estimation"]
estimate_rows = []
for item in candidates["candidates"]:
    for direction in ["denoise", "noise"]:
        group = estimation[(estimation.contrast == item["contrast"]) & (estimation.direction == direction)
                           & (estimation.site == item["site"]) & (estimation.offset == item["offset"])]
        if len(group) == 0:
            continue
        pooled = mi_metrics.pooled_recovery(group["raw_change"], group["m_clean"] - group["m_corrupt"], group["match_id"], N_BOOT)
        raw = mi_metrics.summarise(group["aligned_change"], group["match_id"], N_BOOT)
        estimate_rows.append({"contrast": item["contrast"], "site": item["site"], "offset": item["offset"], "direction": direction,
                              "n": raw["n"], "pooled recovery": pooled["recovery"], "R CI low": pooled["ci_low"],
                              "R CI high": pooled["ci_high"], "mean aligned change": raw["mean"], "CI low": raw["ci_low"], "CI high": raw["ci_high"]})
estimates = pd.DataFrame(estimate_rows)
estimates.to_csv(OUT / "results" / "MB06_candidate_estimates.csv", index=False)
print(estimates.round(3).to_string(index=False) if len(estimates) else "no candidates to estimate")

site_rows = []
for (half, contrast, direction, site, offset), group in sweep.groupby(["half", "contrast", "direction", "site", "offset"], sort=False):
    site_rows.append({"half": half, "contrast": contrast, "direction": direction, "site": site, "offset": offset,
                      "pooled recovery": mi_metrics.pooled_ratio(group["raw_change"], group["m_clean"] - group["m_corrupt"]),
                      "mean raw change": group["raw_change"].mean()})
site_summary = pd.DataFrame(site_rows)
site_summary.to_csv(OUT / "results" / "MB06_site_summary.csv", index=False)

        contrast          site  offset direction   n  pooled recovery  R CI low  R CI high  mean aligned change  CI low  CI high
         context     embedding      -9   denoise 529           -0.003    -0.005     -0.000               -0.018  -0.030   -0.001
         context     embedding      -9     noise 529            0.001    -0.000      0.001                0.004  -0.002    0.008
         context     embedding       0   denoise 529            0.980     0.964      0.994                6.378   5.994    6.835
         context     embedding       0     noise 529            1.004     0.985      1.020                6.536   6.138    6.971
         context     L0 head 0     -13   denoise 529           -0.000    -0.000      0.000               -0.001  -0.002    0.001
         context     L0 head 0     -13     noise 529            0.000    -0.000      0.000                0.001  -0.001    0.003
         context    L0 mlp_out      -8   denoise 529            0.001    -0.001      0.002       

In [9]:
site_order = [label for label, _, _ in SITES]
for contrast in CONTRASTS:
    fig, axes = plt.subplots(2, 2, figsize=(16 if contrast == "context" else 9, 9))
    for row, direction in enumerate(["denoise", "noise"]):
        for col, half in enumerate(["discovery", "estimation"]):
            frame = site_summary[(site_summary.contrast == contrast) & (site_summary.direction == direction) & (site_summary.half == half)]
            grid = frame.pivot(index="site", columns="offset", values="pooled recovery").reindex(site_order)
            axis = axes[row, col]
            image = axis.imshow(grid.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
            axis.set_yticks(range(len(site_order))); axis.set_yticklabels(site_order, fontsize=7)
            axis.set_xticks(range(grid.shape[1])); axis.set_xticklabels(grid.columns, fontsize=7)
            axis.set_xlabel("position relative to the target"); axis.set_title(f"{contrast} | {direction} | {half}", fontsize=9)
            for item in candidates["candidates"]:
                if item["contrast"] == contrast and item["site"] in site_order and item["offset"] in list(grid.columns):
                    axis.add_patch(plt.Rectangle((list(grid.columns).index(item["offset"]) - 0.5, site_order.index(item["site"]) - 0.5),
                                                 1, 1, fill=False, edgecolor="black", linewidth=1.5))
            plt.colorbar(image, ax=axis, fraction=0.03)
    plt.tight_layout(); plt.savefig(OUT / "figures" / f"MB06_{contrast.replace(' ', '_')}.png", bbox_inches="tight"); plt.show()

/tmp/ipykernel_304270/3120772847.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / f"MB06_{contrast.replace(' ', '_')}.png", bbox_inches="tight"); plt.show()


/tmp/ipykernel_304270/3120772847.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / f"MB06_{contrast.replace(' ', '_')}.png", bbox_inches="tight"); plt.show()


/tmp/ipykernel_304270/3120772847.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / f"MB06_{contrast.replace(' ', '_')}.png", bbox_inches="tight"); plt.show()


**How to read the heatmaps.** Rows are sites from the embedding (top) to the last layer; columns are positions relative to the target. Colour is pooled recovery: red = patching this activation restores (denoising) or removes (noising) the clean behaviour; blue = it moves Δ the other way. Outlined cells are the discovery-half candidates. A candidate is only credible if its estimation-half cell has the same colour.

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 7. Controls on the sweep

1. **Matched non-conflict events** under the identity-removal contrast (denoising, target position): tells arbitration-specific sites apart from sites that carry identity on every event.
2. **Same-player transfer:** for identity removal, the *change that the identity input causes* at a site is measured on *another* event of the same player (clean minus identity-removed activation at that event's own target) and added to this event's identity-removed run. Copying the whole activation would also carry the other event's context; transferring only the identity-induced difference isolates the identity content. At the embedding the difference is the player's own vector, so recovery there must equal the own-event value (a built-in check). High recovery at a later site means the identity content there is player-general; low transfer recovery with high own-event recovery means the site's identity content depends on the event's context.

In [10]:
control_rows = []
with torch.no_grad():
    if len(controls):
        for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, controls, BATCH):
            chunk = controls.iloc[part]
            corrupted = interventions.remove_feature(inputs, IDENTITY, positions)
            m_clean, m_corrupt, patched = patching.patch_batch(MODEL, inputs, corrupted, positions, a_ctx, a_id, SITES, [0], ["denoise"])
            for (direction, label, offset), values in patched.items():
                control_rows.append(pd.DataFrame({"match_id": chunk["match_id"].to_numpy(), "site": label, "m_clean": m_clean,
                                                  "m_corrupt": m_corrupt, "m_patched": values}))
control_sweep = pd.concat(control_rows) if control_rows else pd.DataFrame()

pairs = []
for player, group in population.groupby("player"):
    if len(group) < 2:
        continue
    rows = group.index.to_numpy()
    for i, index in enumerate(rows):
        pairs.append((index, rows[(i + 1) % len(rows)]))
transfer_rows = []
if pairs:
    pair_frame = pd.DataFrame(pairs, columns=["target_index", "source_index"])
    target_table = population.loc[pair_frame["target_index"]].reset_index(drop=True)
    source_table = population.loc[pair_frame["source_index"]].reset_index(drop=True)
    site_names = sorted(set(site for _, site, _ in SITES))
    with torch.no_grad():
        for part in mi_setup.batch_slices(len(target_table), BATCH):
            t_chunk, s_chunk = target_table.iloc[part], source_table.iloc[part]
            inputs, _ = mi_setup.window_inputs(bundle, val_view, t_chunk["window"].to_numpy())
            source_inputs, _ = mi_setup.window_inputs(bundle, val_view, s_chunk["window"].to_numpy())
            positions = torch.as_tensor(t_chunk["position"].to_numpy(), device=DEVICE)
            source_positions = torch.as_tensor(s_chunk["position"].to_numpy(), device=DEVICE)
            a_ctx = torch.as_tensor(t_chunk["a_ctx"].to_numpy(), device=DEVICE); a_id = torch.as_tensor(t_chunk["a_id"].to_numpy(), device=DEVICE)
            corrupted = interventions.remove_feature(inputs, IDENTITY, positions)
            clean_logits = hooks.forward(MODEL, inputs)[0]; corrupt_logits = hooks.forward(MODEL, corrupted)[0]
            m_clean = mi_metrics.delta(mi_metrics.at_target(clean_logits, positions), a_ctx, a_id)
            m_corrupt = mi_metrics.delta(mi_metrics.at_target(corrupt_logits, positions), a_ctx, a_id)
            source_corrupted = interventions.remove_feature(source_inputs, IDENTITY, source_positions)
            _, source_clean = hooks.forward(MODEL, source_inputs, capture=site_names)
            _, source_corrupt = hooks.forward(MODEL, source_corrupted, capture=site_names)
            index = torch.arange(len(positions), device=DEVICE)
            for label, site, head in SITES:
                difference = source_clean[site][index, source_positions] - source_corrupt[site][index, source_positions]
                if head is not None:
                    keep = torch.zeros_like(difference); keep[:, head] = difference[:, head]; difference = keep
                shift = torch.zeros_like(source_clean[site])
                shift[index, positions] = difference
                logits, _ = hooks.forward(MODEL, corrupted, replace={site: hooks.add_at(shift)})
                transfer_rows.append(pd.DataFrame({"match_id": t_chunk["match_id"].to_numpy(), "site": label, "m_clean": m_clean,
                                                   "m_corrupt": m_corrupt,
                                                   "m_patched": mi_metrics.delta(mi_metrics.at_target(logits, positions), a_ctx, a_id)}))
transfer = pd.concat(transfer_rows) if transfer_rows else pd.DataFrame()

def pooled_by_site(frame):
    out = {}
    for site, group in frame.groupby("site", sort=False):
        out[site] = mi_metrics.pooled_ratio(group["m_patched"] - group["m_corrupt"], group["m_clean"] - group["m_corrupt"])
    return out
own = sweep[(sweep.contrast == "identity removal") & (sweep.direction == "denoise") & (sweep.offset == 0)]
comparison = pd.DataFrame({"conflict events (own activation)": pooled_by_site(own)})
if len(control_sweep):
    comparison["matched controls"] = pd.Series(pooled_by_site(control_sweep))
if len(transfer):
    comparison["same-player transfer"] = pd.Series(pooled_by_site(transfer))
comparison = comparison.reindex(site_order)
comparison.to_csv(OUT / "results" / "MB06_controls.csv")
print("Pooled recovery, identity removal, denoising, target position:")
print(comparison.round(3).to_string())

mi_setup.save_json(OUT / "results" / "MB06_summary.json", {"candidates": candidates, "estimates": estimates.to_dict(orient="records"),
                   "controls": comparison.reset_index().to_dict(orient="records"), "run_mode": RUN_MODE})
fingerprint.record_files(MANIFEST, {"results/MB06_summary.json": OUT / "results" / "MB06_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB06", "experiment": "E12 activation patching", "n_candidates": len(candidates["candidates"]),
                                "identity_variable": IDENTITY, "seed": PRIMARY_SEED, "run_mode": RUN_MODE})

Pooled recovery, identity removal, denoising, target position:
               conflict events (own activation)  matched controls  same-player transfer
embedding                                 1.000             1.000                 1.000
L0 attn_out                               0.051             0.050                 0.024
L0 head 0                                 0.023             0.039                 0.001
L0 head 1                                 0.028             0.011                 0.022
L0 mlp_out                                0.618             0.616                 0.430
L0 resid_post                             1.000             1.000                 0.797
L1 attn_out                               0.002            -0.013                -0.019
L1 head 0                                -0.003            -0.017                -0.013
L1 head 1                                 0.005             0.004                -0.006
L1 mlp_out                                0.519          

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):